# Prompt 06: Structured Output Prompting

1. Compare weak vs rich field descriptions
2. Order fields to encourage reasoning-before-answer
3. Hybrid free-text + structured output
4. Enum enforcement demo
5. 'Reason for missing' pattern
6. Exercises: build a production schema for one feature

Deps: `pip install anthropic openai pydantic`

In [ ]:
import os, json, re

def call(system, user, max_tokens=500):
    if os.getenv('ANTHROPIC_API_KEY'):
        from anthropic import Anthropic
        r = Anthropic().messages.create(model='claude-sonnet-4-6', max_tokens=max_tokens, temperature=0,
            system=system, messages=[{'role':'user','content':user}])
        return r.content[0].text
    if os.getenv('OPENAI_API_KEY'):
        from openai import OpenAI
        r = OpenAI().chat.completions.create(model='gpt-4o-mini', max_tokens=max_tokens, temperature=0,
            messages=[{'role':'system','content':system},{'role':'user','content':user}])
        return r.choices[0].message.content
    return '[no provider]'

## 1. Weak vs Rich Descriptions

In [ ]:
text = 'Bill from: Acme Labs LLC. Total: $1,234.56 (taxes incl.). Due by 12 Apr 2026.'

weak_schema = '''Return JSON:
{"vendor": str, "total": number, "due": str}'''

rich_schema = '''Return JSON matching this schema EXACTLY:

{
  "vendor":  string,   // Legal vendor name as printed on the invoice ("Acme Labs LLC", not "Acme"). Do NOT translate or abbreviate.
  "total":   number,   // Total amount due in USD. Numeric only; do NOT include a currency symbol. If ambiguous, use the largest dollar figure shown.
  "due":     string    // Due date in ISO-8601 form (YYYY-MM-DD). If the input has a different format, convert it.
}'''

for name, sch in [('weak', weak_schema), ('rich', rich_schema)]:
    out = call('You extract structured data. Return JSON only, no prose.', f'{sch}\n\nText:\n{text}')
    print(f'[{name}]', out)
    try:
        print('  parsed:', json.loads(re.search(r'\{.*\}', out, re.S).group(0)))
    except Exception as e:
        print('  parse failed:', e)
    print()

## 2. Order Fields for Reasoning-Before-Answer

The first schema puts `answer` last, forcing the model to collect evidence first.

In [ ]:
doc = '''Policy 4.3: Full-time employees accrue 15 days PTO per year. 
Policy 4.4: PTO may be rolled over up to 5 days.  
Policy 4.5: PTO does not accrue during unpaid leave.'''

q = 'If I took 4 weeks of unpaid leave, how does that affect my PTO this year?'

reason_first = '''Return JSON:
{
  "citations":  array of policy IDs used (e.g., ["4.3","4.5"]),
  "reasoning":  string (short, using only the citations),
  "answer":     string (the final answer),
  "confidence": "low"|"medium"|"high"
}'''

answer_first = '''Return JSON:
{
  "answer":     string (the final answer),
  "reasoning":  string,
  "citations":  array of policy IDs
}'''

for name, sch in [('reason-first', reason_first), ('answer-first', answer_first)]:
    out = call('You answer from the provided context only. Return JSON.', f'{sch}\n\nContext:\n{doc}\n\nQ: {q}')
    print(f'=== {name} ===')
    print(out)
    print()

Notice: the reason-first schema typically produces more grounded, citation-first answers. Answer-first often produces confident answers and weaker citations afterward.

## 3. Hybrid Free-Text + Structured

In [ ]:
hybrid = '''Return TWO sections exactly:

<analysis>
Prose analysis for a human reviewer. Use markdown. 2-4 sentences.
</analysis>

<extracted>
JSON only: { "vendor": string, "total_usd": number, "due": "YYYY-MM-DD" }
</extracted>
'''

raw = call('You extract and summarize invoices.', f'{hybrid}\n\nInvoice:\n{text}')
print(raw)

analysis = re.search(r'<analysis>(.*?)</analysis>', raw, re.S).group(1).strip()
extracted = json.loads(re.search(r'<extracted>.*?(\{.*\}).*?</extracted>', raw, re.S).group(1))
print('\n--- analysis (show to user) ---\n', analysis)
print('\n--- extracted (send downstream) ---\n', extracted)

## 4. Enum Enforcement

In [ ]:
without_enum = '''Return JSON: {"status": string}.\nThe ticket says: "customer canceled their subscription today."'''
with_enum    = '''Return JSON: {"status": one of ["open","in_progress","resolved","canceled"]}.
If the value is not obviously one of these, pick the closest and record it as-is.\n\nThe ticket says: "customer canceled their subscription today."'''

print('without enum:', call('Return JSON only.', without_enum))
print('with enum:   ', call('Return JSON only.', with_enum))

## 5. Reason-for-Missing Pattern

In [ ]:
input_text = 'Bill from Acme. Total: $500. Please pay promptly.'   # no due date

without = '''Return JSON: {"vendor": string, "total": number, "due": "YYYY-MM-DD" or null}.'''
withpat = '''Return JSON: {
  "vendor": string,
  "total": number,
  "due": "YYYY-MM-DD" or null,
  "due_source": "explicit|inferred|not_found"   // how you determined the due date.
}
Do NOT infer or estimate dates not in the text. If not explicit, set due=null, due_source="not_found".'''

print('without pattern:', call('Return JSON only.', f'{without}\n\n{input_text}'))
print('with pattern:   ', call('Return JSON only.', f'{withpat}\n\n{input_text}'))

## 6. Exercise: Production Schema for One Feature

Pick one real extraction task. Produce:

- A Pydantic model with `Field(description="...")` on every field.
- `ordered_fields` so reasoning comes before answer.
- Enums wherever categorical.
- `_source` sibling fields for fields that might be absent.
- 20 labeled test inputs including edge cases.

Compare strict JSON mode vs XML tags on the same inputs. Record which produces higher schema-validation success. That rate is your production SLI.

## 7. Takeaways

- **Rich descriptions** do more than strict types.
- **Order** fields to induce reason-before-answer.
- **Hybrid prose + structured** tags give the best of both worlds.
- **Enumerate enums** to prevent invented labels.
- **`_source` siblings** prevent silent fabrication on nullable fields.

Next: **07 — Prompt Templates & Variables**.